# Proceso ETL a Snowflake

Este notebook lee archivos CSV desde la carpeta `data`, infiere su esquema y carga los datos en Snowflake preservando los nombres de columnas originales.

In [17]:
# Instalación de librerías necesarias (si no están instaladas)
%pip install pandas snowflake-connector-python python-dotenv

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [18]:
import pandas as pd
import snowflake.connector
import os
from dotenv import load_dotenv
import numpy as np

# Cargar variables de entorno desde el archivo .env
load_dotenv()

True

In [19]:
# Configuración de la conexión a Snowflake usando variables de entorno
ACCOUNT = os.getenv('ACCOUNT')
USER = os.getenv('USER')
PASSWORD = os.getenv('PASSWORD')
WAREHOUSE = os.getenv('WAREHOUSE')
DATABASE = os.getenv('DATABASE')
SCHEMA = os.getenv('SCHEMA')
ROLE = os.getenv('ROLE')

try:
    conn = snowflake.connector.connect(
        user=USER,
        password=PASSWORD,
        account=ACCOUNT,
        warehouse=WAREHOUSE,
        database=DATABASE,
        schema=SCHEMA,
        role=ROLE
    )
    cursor = conn.cursor()
    print("Conexión exitosa a Snowflake")
except Exception as e:
    print(f"Error al conectar: {str(e)}")
    print("Asegúrate de haber configurado correctamente el archivo .env")

Conexión exitosa a Snowflake


In [20]:
def map_pandas_dtype_to_snowflake(dtype):
    """Mapea tipos de datos de pandas a tipos de Snowflake"""
    if pd.api.types.is_integer_dtype(dtype):
        return 'INTEGER'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BOOLEAN'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'TIMESTAMP_NTZ'
    else:
        return 'VARCHAR'

In [21]:
def create_table_sql(df, table_name, constraints=None):
    """Genera la sentencia CREATE TABLE IF NOT EXISTS basada en el dataframe y restricciones opcionales"""
    columns = []
    for col in df.columns:
        dtype = df[col].dtype
        sf_type = map_pandas_dtype_to_snowflake(dtype)
        columns.append(f'"{col}" {sf_type}')
    
    if constraints:
        if 'pk' in constraints:
            pk_cols = [f'"{col}"' for col in constraints['pk']]
            pk_str = ", ".join(pk_cols)
            columns.append(f'PRIMARY KEY ({pk_str})')
        
        if 'fk' in constraints:
            for fk in constraints['fk']:
                columns.append(f'FOREIGN KEY ("{fk["col"]}") REFERENCES {fk["ref_table"].upper()} ("{fk["ref_col"]}")')

    columns_sql = ", ".join(columns)
    # Usamos IF NOT EXISTS para no borrar datos previos
    create_stmt = f"CREATE TABLE IF NOT EXISTS {table_name.upper()} ({columns_sql})"
    return create_stmt

def create_staging_table_sql(df, table_name):
    """Genera la sentencia CALREATE OR REPLACE TEMPORARY TABLE para STAGING"""
    columns = []
    for col in df.columns:
        dtype = df[col].dtype
        sf_type = map_pandas_dtype_to_snowflake(dtype)
        columns.append(f'"{col}" {sf_type}')
    
    columns_sql = ", ".join(columns)
    stg_table_name = f"{table_name}_STG"
    # TEMPORARY: La tabla solo existe durante la sesión y no se guarda permanentemente
    create_stmt = f"CREATE OR REPLACE TEMPORARY TABLE {stg_table_name.upper()} ({columns_sql})"
    return create_stmt

def generate_merge_sql(table_name, df, pk_columns):
    """Genera la sentencia MERGE INTO para upsert"""
    stg_table_name = f"{table_name}_STG"
    
    # Condición ON
    on_conditions = []
    for pk in pk_columns:
        on_conditions.append(f'T."{pk}" = S."{pk}"')
    on_clause = " AND ".join(on_conditions)
    
    # Columnas para INSERT
    cols = [f'"{col}"' for col in df.columns]
    cols_str = ", ".join(cols)
    vals_str = ", ".join([f'S."{col}"' for col in df.columns])
    
    merge_stmt = f"""
    MERGE INTO {table_name.upper()} AS T
    USING {stg_table_name.upper()} AS S
    ON {on_clause}
    WHEN NOT MATCHED THEN
        INSERT ({cols_str}) VALUES ({vals_str})
    """
    return merge_stmt

In [22]:
def insert_data_cursor(df, table_name, cursor):
    """Inserta datos usando el cursor y executemany"""
    # Preparar sentencia INSERT
    # Usamos comillas dobles para coincidir con la definición de tabla
    cols = [f'"{col}"' for col in df.columns]
    placeholders = ["%s"] * len(cols)
    cols_sql = ", ".join(cols)
    placeholders_sql = ", ".join(placeholders)
    
    insert_stmt = f"INSERT INTO {table_name.upper()} ({cols_sql}) VALUES ({placeholders_sql})"
    
    # Convertir NaN a None para que SQL lo interprete como NULL
    data = df.where(pd.notnull(df), None).values.tolist()
    
    print(f"Insertando {len(data)} filas en {table_name}...")
    cursor.executemany(insert_stmt, data)
    print("Inserción completada.")

In [23]:
# Carpeta donde están los CSVs
DATA_FOLDER = 'data'

# Definición de restricciones (Constraints)
TABLE_CONSTRAINTS = {
    'clientes': {
        'pk': ['id_cliente']
    },
    'productos': {
        'pk': ['id_producto']
    },
    'sucursales': {
        'pk': ['id_sucursal']
    },
    'empleados': {
        'pk': ['id_vendedor'],
        'fk': [
            {'col': 'sucursal', 'ref_table': 'sucursales', 'ref_col': 'id_sucursal'}
        ]
    },
    'facturas': {
        'pk': ['num_factura', 'fecha_venta', 'producto'],
        'fk': [
            {'col': 'producto', 'ref_table': 'productos', 'ref_col': 'id_producto'},
            {'col': 'vendedor', 'ref_table': 'empleados', 'ref_col': 'id_vendedor'},
            {'col': 'cliente', 'ref_table': 'clientes', 'ref_col': 'id_cliente'}
        ]
    }
}

ORDERED_TABLES = ['sucursales', 'clientes', 'productos', 'empleados', 'facturas']

if not os.path.exists(DATA_FOLDER):
    print(f"La carpeta '{DATA_FOLDER}' no existe. Creándola...")
    os.makedirs(DATA_FOLDER)

for table_name in ORDERED_TABLES:
    file = f"{table_name}.csv"
    try:
        print(f"\nProcesando {file}...")
        file_path = os.path.join(DATA_FOLDER, file)
        
        if not os.path.exists(file_path):
            print(f"  -> Archivo {file} no encontrado, saltando...")
            continue

        # 1. Cargar CSV
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            print(f"  -> Error de encoding utf-8 en {file}, intentando con latin1...")
            df = pd.read_csv(file_path, encoding='latin1')
        
        constraints = TABLE_CONSTRAINTS.get(table_name)

        # 2. Crear tabla DESTINO si no existe
        create_sql = create_table_sql(df, table_name, constraints)
        cursor.execute(create_sql)
        
        # 3. Crear tabla STAGING (Temporal)
        stg_sql = create_staging_table_sql(df, table_name)
        # print(f"Creando staging temporal para {table_name}...")
        cursor.execute(stg_sql)
        
        # 4. Poblar tabla STAGING
        stg_table_name = f"{table_name}_STG"
        insert_data_cursor(df, stg_table_name, cursor)
        
        # 5. MERGE de Staging a Destino
        if constraints and 'pk' in constraints:
            # print(f"Merge de nuevos registros en {table_name}...")
            merge_sql = generate_merge_sql(table_name, df, constraints['pk'])
            cursor.execute(merge_sql)
            print(f"Registros procesados en {table_name}.")
        else:
            print(f"ADVERTENCIA: No se definieron PK para {table_name}, no se puede hacer MERGE seguro.")
        
        # 6. Limpieza explícita (aunque sea temporal)
        cursor.execute(f"DROP TABLE IF EXISTS {stg_table_name.upper()}")
        
        conn.commit()
        
    except Exception as e:
        print(f"Error procesando {file}: {str(e)}")
        if 'conn' in locals():
            conn.rollback()


Procesando sucursales.csv...
  -> Error de encoding utf-8 en sucursales.csv, intentando con latin1...
Insertando 5 filas en sucursales_STG...
Inserción completada.
Registros procesados en sucursales.

Procesando clientes.csv...
Insertando 120 filas en clientes_STG...
Inserción completada.
Registros procesados en clientes.

Procesando productos.csv...
Insertando 37 filas en productos_STG...
Inserción completada.
Registros procesados en productos.

Procesando empleados.csv...
Insertando 50 filas en empleados_STG...
Inserción completada.
Registros procesados en empleados.

Procesando facturas.csv...
Insertando 11962 filas en facturas_STG...
Inserción completada.
Registros procesados en facturas.


In [24]:
# Cerrar conexión
if 'cursor' in locals():
    cursor.close()
if 'conn' in locals():
    conn.close()
print("\nConexión cerrada.")


Conexión cerrada.
